# Chapter 20 — Multi-GPU with NCCL & ZeRO

> Course: **llm.c — Zero to Hero**, Chapter 20 of ~20.  **Final chapter.**
> Builds on Chapters 8 (parameters/grads/optimizer state) and 18 (GPU AdamW).

GPT-2 124M fits easily on one GPU. **GPT-3 175B** does not — its FP16 weights alone are 350 GB, plus 1.4 TB for AdamW master + m + v. You need many GPUs, working together. This chapter is about how `llm.c` does that.

We'll cover:

1. **DDP (Distributed Data Parallel)** — the simplest multi-GPU strategy: each GPU has a full copy of the model, processes its own batch shard, and gradients are *all-reduced* across GPUs.
2. **NCCL** — NVIDIA's communication library. Like MPI, but tuned for GPU↔GPU transfers via NVLink/PCIe.
3. **ZeRO-1** — the optimizer-state-sharding trick that **divides** master weights, m, and v across GPUs. Cuts optimizer memory by N for N GPUs.

You won't run multi-GPU code here (single-GPU box), but you'll read `llmc/zero.cuh` and understand exactly what every line does.

### Learning objectives

By the end of this chapter you will:

- Explain DDP and why it doesn't reduce memory per GPU.
- Explain ZeRO-1 and how it changes the optimizer memory profile.
- Read `llmc/zero.cuh::multi_gpu_async_reduce_gradient` and identify the all-reduce vs reduce-scatter paths.
- Understand the sequencing: gradient sync ↔ optimizer step ↔ parameter broadcast.


## 1. Concept — Distributed Data Parallel (DDP)

Eight GPUs, identical models. Each GPU gets a fraction of the global batch.

```
GPU 0:   forward(x[0:B/8]) → loss → backward → grad_0
GPU 1:   forward(x[B/8:2B/8]) → loss → backward → grad_1
...
GPU 7:   forward(x[7B/8:B]) → loss → backward → grad_7

NCCL all-reduce: every GPU now has  (grad_0 + grad_1 + ... + grad_7)
Optimizer step: each GPU runs the same AdamW on the same averaged grads
```

The all-reduce is the only inter-GPU communication. Each GPU still holds a **full copy of the model + optimizer state**. So memory per GPU is unchanged from single-GPU; you just get **N× compute** and **N× effective batch size**.

For GPT-2 small (1.5 GB total state), this is fine. For GPT-3 (2 TB total state), it's hopeless — each GPU only has 80 GB. Hence ZeRO.


## 2. Concept — NCCL Primitives

NCCL (NVIDIA Collective Communications Library) provides MPI-like collectives, GPU-aware. The four `llm.c` uses:

| Op | What it does | When |
|---|---|---|
| `ncclAllReduce` | Sum a buffer across all ranks; every rank gets the sum | Plain DDP gradient sync |
| `ncclReduceScatter` | Sum a buffer across all ranks; rank `r` gets only **its slice** | ZeRO-1 gradient sync |
| `ncclAllGather` | Concatenate slices across ranks; everyone gets the full buffer | ZeRO-1 weight broadcast after optimizer step |
| `ncclBroadcast` | Send rank-0's buffer to all others | Init: distribute initial weights |

All of these run on the GPU and use NVLink/PCIe directly. No host involvement, no serialization, no extra memory copies. On an H100 NVLink island, all-reduce of a 1 GB buffer takes ~10 ms.

NCCL setup in `llm.c`:

```cpp
// from llmc/zero.cuh, paraphrased
MultiGpuConfig multi_gpu_config_init(int num_processes, int process_rank, ...) {
    ncclUniqueId nccl_id = get_nccl_id_via_tcp(...);   // each rank uses same id
    ncclCommInitRank(&result.nccl_comm, num_processes, nccl_id, process_rank);
    cudaStreamCreate(&result.nccl_stream);
    return result;
}
```

Each process has the same `nccl_id`, so they all join the same communicator. Subsequent `ncclAllReduce` etc. operate over that communicator.


## 3. Concept — ZeRO Memory Math

Recall from Chapter 8 that optimizer state is `3 × num_params` of FP32 (master, m, v) plus the BF16 working copy. Per GPU:

| Storage | DDP (no ZeRO) | ZeRO-1 (8 GPUs) |
|---|---|---|
| Working params (BF16) | 250 MB | 250 MB (replicated) |
| Master params (FP32)  | 500 MB | 500 / 8 = 62.5 MB |
| Adam `m` (FP32)        | 500 MB | 500 / 8 = 62.5 MB |
| Adam `v` (FP32)        | 500 MB | 500 / 8 = 62.5 MB |
| Gradients (BF16)      | 250 MB | 250 MB (replicated for now)* |
| **Total** for GPT-2 124M | **2 GB** | **0.6 GB** |

*ZeRO-2 also shards gradients, ZeRO-3 also shards parameters. `llm.c` implements ZeRO-1 only.

The big win: **optimizer state is `3×` the size of the model**, so sharding it saves a lot. Working params + gradients are needed everywhere on the GPU during forward/backward, so they can't be sharded the same way.


## 4. Walkthrough — `multi_gpu_async_reduce_gradient`

From [`llmc/zero.cuh`](llmc/zero.cuh) (lines 510-555, paraphrased):

```cpp
void multi_gpu_async_reduce_gradient(floatX* grad_memory, size_t total_parameters,
                                      MultiGpuConfig* config, cudaStream_t compute_stream) {
    if (config->num_processes == 1) return;     // single GPU, nothing to do

    cudaStreamWaitEvent(config->nccl_stream, ...);    // wait for compute to finish

    if (config->zero_stage == 0) {
        // DDP: every GPU gets the full summed gradients
        ncclAllReduce(grad_memory, grad_memory, total_parameters, ncclFloatX, ncclSum,
                      config->nccl_comm, config->nccl_stream);

    } else if (config->zero_stage == 1) {
        // ZeRO-1: each GPU only gets its shard of the summed gradients
        ShardInfo shard = multi_gpu_get_shard_offset(total_parameters, config, 1);
        ncclReduceScatter(grad_memory, grad_memory + shard.offset, shard.size,
                          ncclFloatX, ncclSum, config->nccl_comm, config->nccl_stream);
    }
}
```

The pivot: `zero_stage == 0` calls `ncclAllReduce` (everyone gets full sum); `zero_stage == 1` calls `ncclReduceScatter` (each rank gets its slice). The slice is determined by `multi_gpu_get_shard_offset`, which divides `total_parameters` evenly across processes.

After this call, in ZeRO-1 land, each rank has only `grads[shard.offset : shard.offset + shard.size]`. The rest of `grad_memory` is *garbage* on this rank — but it's about to be overwritten by AdamW operating on this rank's own slice.


## 5. Walkthrough — ZeRO-1 Optimizer Step

After gradient reduce-scatter, each rank runs AdamW *only on its shard*:

```cpp
// pseudocode of what gpt2_update does in ZeRO-1 mode
ShardInfo shard = multi_gpu_get_shard_offset(num_parameters, config, 1);

// each rank only has master/m/v for its shard. AdamW kernel runs over shard.size params.
adamw_kernel<<<grid, block>>>(
    master_memory + shard.offset,
    m_memory + shard.offset,
    v_memory + shard.offset,
    grad_memory + shard.offset,
    shard.size,
    lr, beta1, beta2, eps, wd, t);

// after the AdamW, master_memory holds the updated params for THIS rank's shard.
// We need to broadcast updated params back to everyone, in BF16 working form.
ncclAllGather(master_memory + shard.offset,    // local shard (FP32)
              params_memory_bf16,               // full buffer to fill (BF16)
              shard.size,
              ncclFloatX, config->nccl_comm, config->nccl_stream);
// (in practice the cast and gather are fused / done in stages)
```

So the per-step communication for ZeRO-1 is:

1. `ReduceScatter`(grads) → each rank has its slice of the summed grads
2. AdamW on the slice → master, m, v updated locally for this slice
3. `AllGather`(updated master) → every rank gets the full new BF16 params

DDP needs only step 1's all-reduce. ZeRO-1 trades **one extra all-gather per step** for **8× less optimizer memory**. The communication volume is *lower* than DDP, actually — ReduceScatter + AllGather = 2× model size vs. DDP's AllReduce = 2× model size, same total. The win is purely memory.


## 6. The Training Loop with Multi-GPU

`train_gpt2.cu`'s training loop with ZeRO-1 enabled:

```cpp
for (int step = 1; step <= num_iterations; step++) {
    gpt2_zero_grad(model);

    for (int micro = 0; micro < grad_accum_steps; micro++) {
        dataloader_next_batch(loader);            // gets THIS rank's micro-batch shard
        gpt2_forward(model, ...);
        gpt2_backward_and_reduce(model, ...);
        // grads accumulated locally, no comm yet
    }

    // ONE communication round per outer step (after K micro-batches):
    multi_gpu_async_reduce_gradient(model->grads_memory, num_parameters, &config, stream);
    // → ReduceScatter (ZeRO-1) or AllReduce (DDP)

    // gradient norm computation also needs to be all-reduced! (norm² is summed across ranks)
    gpt2_compute_grad_norm(model, &config);

    // local AdamW on this rank's shard
    gpt2_update(model, lr, beta1, beta2, eps, wd, step, max_grad_norm);

    // broadcast new params (in ZeRO-1)
    multi_gpu_allgather_params(model, &config);
}
```

The key insight: **all the per-layer kernels you've studied run unchanged**. The multi-GPU machinery is purely outer orchestration — a few NCCL calls plus a slice offset for the AdamW loop. Eight GPUs work, with no changes to attention, layernorm, matmul, etc.


## 7. Running Multi-GPU `llm.c`

For completeness — to actually train multi-GPU on a real machine:

```bash
# Build with NCCL support
make train_gpt2cu USE_NCCL=1 USE_CUDNN=1

# Launch on 8 GPUs of one node
mpirun -np 8 ./train_gpt2cu -zs 1     # -zs 1 enables ZeRO-1

# Or across multiple nodes (rank, master IP, etc. via env vars)
```

`mpirun` spawns 8 processes. Each calls `multi_gpu_config_init` to set up its NCCL communicator, then runs the same training loop with the appropriate rank. NCCL handles all GPU↔GPU traffic.

This is what `llm.c`'s authors used to verify they could reproduce real GPT-2 training in pure C/CUDA. With this infrastructure, the same code that trains on your RTX 4080 also trains on a 2048-GPU cluster.


## 8. Translation Bridge

| PyTorch | `llm.c` |
|---|---|
| `torch.nn.parallel.DistributedDataParallel(model)` | DDP via `ncclAllReduce` after backward |
| `torch.distributed.fsdp.FullyShardedDataParallel(model)` | ZeRO-3 (not in `llm.c` — only ZeRO-1) |
| `deepspeed.optim.ZeroOptimizer(optim, stage=1)` | ZeRO-1 via `ncclReduceScatter` + sharded AdamW + `ncclAllGather` |
| `accelerate launch ...` | `mpirun -np N ...` |


## End of Course — Where to From Here?

You've made it. From "what does `int* x` mean?" in Chapter 1 to "ZeRO-1 reduce-scatter into sharded AdamW" in Chapter 20.

A non-exhaustive list of next steps:

- **Reproduce the GPT-2 124M training run.** Download the starter pack, run `./train_gpt2cu`, watch loss go down. Total time on a single 4080: ~2 hours.
- **Run `./test_gpt2cu`.** Confirm every layer matches PyTorch byte-for-byte.
- **Read `train_llama3.py`.** The repo's other reference: same architecture concepts, slightly different details (RoPE, RMSNorm, SiLU). Once you can read GPT-2, Llama-3 is a small step.
- **Fork and modify a kernel.** Pick one — e.g., `layernorm_forward_kernel6`. Replace it with something different (e.g., RMSNorm), and verify against PyTorch via `test_gpt2cu`. This is the best way to consolidate.
- **Read CUTLASS or CUTE.** NVIDIA's matmul-kernel template library. Once you understand `llm.c`'s hand-rolled kernels, CUTLASS shows you the next layer of abstraction.

Or skip all of that and go train a model. You now have the mental model — the rest is engineering.

### Course summary

| Part | Chapters | What you can do now |
|---|---|---|
| I — C foundations | 1-8 | Read every line of `train_gpt2.c`, derive every backward pass, write a CPU AdamW |
| II — CUDA fundamentals | 9-14 | Write CUDA kernels with grid-stride loops, coalesced loads, warp reductions, block reductions, cuBLAS |
| III — Production GPU | 15-20 | Read every line of `train_gpt2.cu`, follow the full mixed-precision training loop end to end |

Thanks for following along. Now go train something.

— *llm.c — Zero to Hero* (Chapters 1-20 complete).
